In [0]:
from databricks.sdk import WorkspaceClient
import requests
import json

w = WorkspaceClient()

app_name = "databrck-weather-vector-search"

# 1. Get the OAuth client ID for this Databricks App
app_client_id = w.apps.get(app_name).oauth2_app_client_id

# 2. Get the notebook's current internal Databricks token
notebook_token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

# 3. Exchange it for a token scoped specifically to this app
token_response = requests.post(
    f"{w.config.host}/oidc/v1/token",
    data={
        "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
        "subject_token": notebook_token,
        "subject_token_type":
            "urn:databricks:params:oauth:token-type:personal-access-token",
        "requested_token_type":
            "urn:ietf:params:oauth:token-type:access_token",
        "scope": "all-apis",
        "audience": app_client_id,
    },
)

token_response.raise_for_status()
app_token = token_response.json()["access_token"]

# 4. Call your Flask API
# Get the app URL dynamically instead of hardcoding
app_info = w.apps.get(app_name)
url = f"{app_info.url}/api/embeddings/ingest"

response = requests.post(
    url,
    headers={
        "Authorization": f"Bearer {app_token}",
        "Content-Type": "application/json",
    },
)

print("Status:", response.status_code)
print(json.dumps(response.json(), indent=2))

In [0]:
Run below sql in Lakebase SQL editor to test the outcomes - 

-- Validate document to embedding is one to many relations
select count(*) from weather_documents;

select count(*) from weather_embeddings;

-- Look at chunking
select a.id,count(b.id) 
from weather_documents a
left join weather_embeddings b on b.document_id = a.id
  group by  a.id
  order by 2 desc;

-- Negative test ; should be Zero
select count(*) from weather_documents where id not in (select document_id from  weather_embeddings )
;

-- Validate Chunk index, and vector dimensions
SELECT
    a.document_id,
    a.chunk_index,
    LENGTH(a.chunk_text) AS chunk_length,
    a.created_at
  , vector_dims(embedding) AS dims
FROM weather_embeddings a
WHERE EXISTS (
    SELECT 1
    FROM weather_documents b
    WHERE LENGTH(b.narrative_text) > 800
      AND a.document_id = b.id
)
ORDER BY a.document_id, a.chunk_index;


-- Validate latest embedding after sync, only relevant document which are latest synced should be updated.
SELECT wd.id, wd.synced_at, MAX(we.created_at) AS latest_embedding_at
FROM weather_documents wd
LEFT JOIN weather_embeddings we
  ON we.document_id = wd.id
 AND we.model_name = 'sentence-transformers/all-MiniLM-L6-v2'
GROUP BY wd.id, wd.synced_at
ORDER BY 3 desc;